**Nombre:** Daniel Soto Villada

**Cédula:** 1000534026

**Correo Institucional:** daniel.soto1@udea.edu.co

**Correo personal:** dnsv.001@gmail.com

#El plano invariable de Laplace del Sistema Solar
El reto consiste en calcular la inclinación del plano invariable de Laplace del Sistema Solar con respecto a la órbita de los planetas del Sistema Solar. Para ello:

* Escoja como fecha, el día de su cumpleaños en $2024$ y como hora las $00:00$ de $UT$.
* Use `astroquery` *(rutina `consulta_horizons` de `pymcel`)* para obtener las posiciones y velocidades del Sol y los 8 planetas mayores del Sistema solar.
* Use `SPICE` para conseguir el valor de los $\mu$ de los cuerpos. Debe tener en cuenta que `SPICE` da el valor de estas cantidades en $km^{3}/s^{2}$ pero para el cálculo usted necesita esa cantidad en m³/s². Haga el cambio de unidades.
* Calcule con esta información el valor de las componentes $x$, $y$, $z$ del momentum angular total del Sistema Solar.
* Calcule el momentum angular de cada planeta y encuentre el ángulo que forma cada uno con respecto al momentum angular total del sistema. Ese ángulo define la orientación del plano de Laplace. Ayuda: el ángulo se puede calcular usando la definición de producto punto.
* Opcional: calcule la latitud y longitud eclíptica del polo del plano de Laplace, es decir, la ubicación del vector del momentum angular total sobre la esfera celeste.

In [ ]:
#@title Instalación de paquetes
!pip install -Uq pymcel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 910.8/910.8 kB 21.6 MB/s eta 0:00:00


In [ ]:
#@title Importación de librerías
import numpy as np
import pymcel as pc
import spiceypy as spy

Paquete pymcel cargado. Versión: 0.6.10


In [ ]:
#@title Descarga de kernels
#Descarga de kernels para obtener las masas
pc.descarga_kernels()
spy.furnsh([
    'pymcel/data/gm_de431.tpc',
    'pymcel/data/de430.bsp',
    'pymcel/data/latest_leapseconds.tls'
])

In [ ]:
#@title Obtención de vectores de estado con `pymcel`

#Etiqueta de los cuerpos del sistema solar
bodies = ['Sun',
           'Mercury Barycenter',
           'Venus Barycenter',
           'Earth-Moon Barycenter',
           'Mars Barycenter',
           'Jupiter Barycenter',
           'Saturn Barycenter',
           'Uranus Barycenter',
           'Neptune Barycenter']

#Listas en las cuales se agrupa la informacion de todos los cuerpos
datoss = [] #[x, y, z, vx, vy, vz] en unidades del SI: m y m/s
muCuerpos = []

#Ciclo para obtener y guardar los vectores de estado y GM de los cuerpos
for id in bodies:
  tabla, df, datos = pc.consulta_horizons(
    id=id,
    location='@0',                #Ubicacion del punto de referencia (SSB)
    epochs='2024-11-15 00:00:00', #Fecha de mi cumpleañoso en el 2024
    datos='vectors'
  )

  datoss.append(datos)
  muCuerpos.append(spy.bodvrd(id,'GM',1)[1][0])

In [ ]:
#@title Array de posiciones y velocidades

N = 9 #Numero total de cuerpos

Rs = np.zeros((N,3))
Vs = np.zeros((N,3))

for i in range(N):
  Rs[i] = datoss[i][0:3]
  Vs[i] = datoss[i][3:]

Obtención de la masa en kg a partir de $GM$

$$M = \frac{1\times 10^{9}}{G}\quad kg$$

In [ ]:
#@title Obtención de las masas a partir de $GM$
G = 6.67191e-11                 #Constante de Gravitacion Universal en N*m^2/kg^2
Ms = np.array(muCuerpos)*1e9/G  #En kg

##Cálculo del momentum angular total de todos lo cuerpos

$$\vec{L} = m\vec{R}\times \vec{V}$$

In [ ]:
#@title Momentum angular total
#L = Momentum angular de cada cuerpo
#Lt = L total
#LtNorm = Norma del momentum angular total

L = np.array([Ms[i]*np.cross(Rs[i],Vs[i]) for i in range(N)])
Lt = L.sum(axis=0)
LtNorm = np.linalg.norm(Lt)

print("Componentes (x, y, z) en [kg*m^2/s] del moentum angular total:\n")
print(f"Lx = {Lt[0]:.3e}\nLy = {Lt[1]:.3e}\nLz = {Lt[2]:.3e}")
print(f"|L| = {LtNorm:.3e}")

Componentes (x, y, z) en [kg*m^2/s] del moentum angular total:

Lx = 8.229e+41
Ly = 2.608e+41
Lz = 3.132e+43
|L| = 3.134e+43


###Valor de las componentes del momentum angular:

$$\vec{L} =
\begin{bmatrix}
L_x \\ L_y \\ L_z
\end{bmatrix} \approx
\begin{bmatrix}
8.229 \times 10^{41} \\
2.608 \times 10^{41} \\
3.132 \times 10^{43}
\end{bmatrix}  \quad kg\cdot m^{2}/s$$

$\quad$

$$|\vec{L}|\approx 3.134 \times 10^{43} \quad kg\cdot m^{2}/s$$

###Cálculo del ángulo que forma el momentum angular de cada cuerpo con el momentum angular total

Usando el producto punto, hallamos que el ángulo que se forma entre el momentum angular de cada cuerpo con el momentum angular total es:
$$\theta_i = \arccos\left(\frac{\vec{L}}{|\vec{L}|}\cdot \frac{\vec{L}_i}{|\vec{L}_i|}\right)$$

In [ ]:
#@title Cálculo de $\theta_i$

theta = np.rad2deg(np.array([np.arccos(np.dot(Lt/np.linalg.norm(Lt), L[i]/np.linalg.norm(L[i]))) for i in range(N)]))
print("Ángulo entre el momentum angular de cada cuerpo con el momentum angular total en grados\n")
for i, id in enumerate(bodies):
  print(f"θ[{id}]:\t {theta[i]:.3f}°")

Ángulo entre el momentum angular de cada cuerpo con el momentum angular total en grados

θ[Sun]:	 0.268°
θ[Mercury Barycenter]:	 6.342°
θ[Venus Barycenter]:	 2.204°
θ[Earth-Moon Barycenter]:	 1.586°
θ[Mars Barycenter]:	 1.685°
θ[Jupiter Barycenter]:	 0.327°
θ[Saturn Barycenter]:	 0.933°
θ[Uranus Barycenter]:	 1.028°
θ[Neptune Barycenter]:	 0.726°


---
#Tabla con los datos obtenidos
---
| Cuerpo   | Inclinación respecto al plano invariable [°] (2024)| Datos de Wikipedia [°] (2009)|
|-----------|------------------------------------------------|---------------------|
| Sol       | 0.268                                          | Sin información |
| Mercurio  | 6.342                                          | 6.34               |
| Venus     | 2.204                                          | 2.19               |
| Tierra    | 1.586                                          | 1.57               |
| Marte     | 1.685                                          | 1.67               |
| Júpiter   | 0.327                                          | 0.32               |
| Saturno   | 0.933                                          | 0.93               |
| Urano     | 1.028                                          | 1.02               |
| Neptuno   | 0.726                                          | 0.72               |

---
*Fuente:* [Invariable plane](https://en.wikipedia.org/wiki/Invariable_plane)




---

